# Overnight Penny-Stock Research Lab: Quick Demo

This notebook reads cached outputs only. It does not initiate market-data or SEC downloads.

**This is research, not investment advice.**

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
outputs = project_root / 'data' / 'outputs'
ranked = pd.read_csv(outputs / 'ranked_candidates.csv')
details = pd.read_csv(outputs / 'ticker_detail_stats.csv')
rejected = pd.read_csv(outputs / 'rejected_tickers.csv')
ranked.head(20)

In [ ]:
# Inspect both calculation modes for the highest-ranked ticker, or the first analyzed ticker.
if not ranked.empty:
    example_ticker = ranked.iloc[0]['ticker']
elif not details.empty:
    example_ticker = details.iloc[0]['ticker']
else:
    example_ticker = None

details.loc[details['ticker'] == example_ticker].T if example_ticker else 'No analyzable ticker output'

In [ ]:
# Reproduce the return formulas from one cleaned cache without downloading data.
if example_ticker:
    prices = pd.read_csv(project_root / 'data' / 'cleaned_prices' / f'{example_ticker}.csv', parse_dates=['Date'])
    prices['overnight_return'] = prices['adj_open'] / prices['adj_close'].shift(1) - 1
    prices['intraday_return'] = prices['adj_close'] / prices['adj_open'] - 1
    prices['close_to_close_return'] = prices['adj_close'] / prices['adj_close'].shift(1) - 1
    display(prices[['Date', 'overnight_return', 'intraday_return', 'close_to_close_return']].tail())
    print('Cumulative overnight:', (1 + prices['overnight_return'].dropna()).prod() - 1)
    print('Cumulative intraday:', (1 + prices['intraday_return'].dropna()).prod() - 1)

In [ ]:
# Audit hard-screen outcomes.
rejected[['ticker', 'rejection_reason', 'details']].head(50)